# FinPulse — Silver Layer
**Purpose:** Clean, validate and enrich Bronze data into Silver tables on Databricks

**Input:** workspace.finpulse.transactions_bronze, workspace.finpulse.stocks_bronze


**Output:** workspace.finpulse.transactions_silver, workspace.finpulse.stocks_silver

In [11]:
import os
from dotenv import load_dotenv
load_dotenv()
from databricks.connect import DatabricksSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
from pathlib import Path

spark = DatabricksSession.builder \
    .host("https://dbc-6ee18cda-1d4e.cloud.databricks.com") \
    .token(os.getenv("DATABRICKS_TOKEN")) \
    .serverless(True) \
    .getOrCreate()

print(f"Spark ready: {spark.version}")

Spark ready: 4.1.0


In [ ]:
def push_large_dataframe(df_pd, target_table, batch_size=500000):
    """
    Pushes a large pandas dataframe to Databricks in batches to avoid timeouts.
    """
    print(f"Pushing {len(df_pd):,} rows to {target_table} in batches of {batch_size:,}...")
    
    total_rows = len(df_pd)
    num_batches = (total_rows + batch_size - 1) // batch_size
    
    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, total_rows)
        batch_pd = df_pd.iloc[start_idx:end_idx]
        
        # Convert batch to spark
        batch_spark = spark.createDataFrame(batch_pd)
        
        mode = "overwrite" if i == 0 else "append"
        batch_spark.write \
            .format("delta") \
            .mode(mode) \
            .saveAsTable(target_table)
            
        print(f"  [Batch {i+1}/{num_batches}] Pushed {len(batch_pd):,} rows")
    
    print(f"DONE: {target_table} push complete")


In [ ]:
def get_latest_partition(base_path):
    p = Path(base_path)
    partitions = sorted([d.name for d in p.iterdir() if d.is_dir() and d.name.startswith('date=')])
    if not partitions:
        raise FileNotFoundError(f"No partitions found in {base_path}")
    return partitions[-1]

TX_SILVER_BASE = r"str(PROJECT_ROOT / 'data' / 'silver' / 'transactions')"
STOCKS_SILVER_BASE = r"str(PROJECT_ROOT / 'data' / 'silver' / 'stocks')"

latest_tx_partition = get_latest_partition(TX_SILVER_BASE)
latest_stocks_partition = get_latest_partition(STOCKS_SILVER_BASE)

print(f"Latest Transactions: {latest_tx_partition}")
print(f"Latest Stocks:       {latest_stocks_partition}")


In [12]:
silver_path = Path(r"str(PROJECT_ROOT / 'data' / 'silver' / 'transactions')\date=2026-04-05")
parquet_files = list(silver_path.glob("*.parquet"))
print(f"Found {len(parquet_files)} parquet files")

df_pandas = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)
print(f"Rows: {len(df_pandas)}")
print(f"Columns: {list(df_pandas.columns)}")

Found 13 parquet files
Rows: 6362604
Columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud', 'ingestion_timestamp', 'data_source', 'pipeline_version', 'is_balance_discrepancy', 'hour_of_simulation', 'day_of_simulation', 'type_encoded', 'risk_flag', 'balance_diff', 'silver_processed_at']


In [13]:
# Fix chunked array issue by resetting and cleaning dtypes
df_pandas = df_pandas.copy()

# Convert any object columns with mixed types to string
for col in df_pandas.columns:
    if df_pandas[col].dtype == 'object':
        df_pandas[col] = df_pandas[col].astype(str)

# Reset index cleanly
df_pandas = df_pandas.reset_index(drop=True)

print("Data cleaned")
print(df_pandas.dtypes)

Data cleaned
step                               int32
type                                 str
amount                           float64
nameOrig                             str
oldbalanceOrg                    float64
newbalanceOrig                   float64
nameDest                             str
oldbalanceDest                   float64
newbalanceDest                   float64
isFraud                            int32
isFlaggedFraud                     int32
ingestion_timestamp       datetime64[ns]
data_source                          str
pipeline_version                     str
is_balance_discrepancy              bool
hour_of_simulation                 int32
day_of_simulation                  int32
type_encoded                       int32
risk_flag                           bool
balance_diff                     float64
silver_processed_at                  str
dtype: object


In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

# Read all parquet files and combine chunks properly
silver_path = Path(r"str(PROJECT_ROOT / 'data' / 'silver' / 'transactions')\date=2026-04-05")
parquet_files = list(silver_path.glob("*.parquet"))

# Read using pyarrow directly instead of pandas
tables = [pq.read_table(f) for f in parquet_files]
combined = pa.concat_tables(tables)
combined = combined.combine_chunks()

print(f"Rows: {combined.num_rows}")
print(f"Columns: {combined.column_names}")

PySparkValueError: [CANNOT_INFER_EMPTY_SCHEMA] Can not infer schema from an empty dataset.

In [ ]:
# Convert pyarrow table to pandas cleanly
df_pandas = combined.to_pandas()
print(f"Pandas rows: {len(df_pandas)}")
print(df_pandas.dtypes)

In [ ]:
print("Pushing Transactions Silver...")
push_large_dataframe(df_pandas, "workspace.finpulse.transactions_silver")


In [ ]:
import pyarrow.parquet as pq
import pyarrow as pa

print("Pushing Stocks Silver...")
stocks_path = Path(STOCKS_SILVER_BASE) / latest_stocks_partition
stocks_files = list(stocks_path.glob("*.parquet"))
stocks_pd = pd.concat([pd.read_parquet(f) for f in stocks_files], ignore_index=True)

push_large_dataframe(stocks_pd, "workspace.finpulse.stocks_silver")
